# Tshivenda Audio Preprocessing

Runs the Step 3 preprocessing pipeline (see `src/`) that turns the NCHLT and ANV/Swivuriso
Tshivenda corpora into the `audio,transcript` CSVs the fine-tuning notebook expects.

Rules implemented (from the EDA findings in `za_next_voices_eda.ipynb`):
- NCHLT: already 16 kHz mono; drop clips < 1 s.
- ANV: resample 48 kHz -> 16 kHz; clips > 30 s excluded from training CSVs
  (transcripts can't be split without forced alignment); optional VAD segmentation
  writes their audio as unlabelled 5-30 s windows.
- Transcripts normalised via `src/text_norm.py` (preserves ḓ ḽ ṅ ṋ ṱ).

**Kernel**: select **MultilingualASR** - it has the ffmpeg@7 library path baked in,
which torchcodec needs to decode audio on macOS.

**Note**: full runs download a lot (NCHLT ~6.3 GB, ANV tens of GB). Set `LIMIT` below
to a small number first to smoke-test, then set it to `None` for the real run.

In [ ]:
import sys
sys.path.insert(0, "../src")

from preprocess_nchlt import process_split as process_nchlt_split
from preprocess_anv import process_split as process_anv_split

LIMIT = 30  # small smoke-test; set to None for the full run

## NCHLT (train / validation / test)

In [ ]:
for split in ["train", "validation", "test"]:
    process_nchlt_split(split, limit=LIMIT)

## ANV / Swivuriso (train / dev / dev_test)

Gated dataset - needs `hf auth login` done once (access already approved).

In [ ]:
for split in ["train", "dev", "dev_test"]:
    process_anv_split(split, limit=LIMIT, segment_long=False)

## Verify the output loads the way the fine-tuning notebook needs

In [ ]:
from datasets import load_dataset, Audio

ds = load_dataset("csv", data_files={
    "train": "../dataset/processed/nchlt_ven/train.csv",
    "dev_test": "../dataset/processed/nchlt_ven/validation.csv",
})
ds = ds.cast_column("audio", Audio(sampling_rate=16000))

row = ds["train"][0]
samples = row["audio"].get_all_samples()
print("splits:", {k: len(v) for k, v in ds.items()})
print("sample rate:", samples.sample_rate)
print("transcript:", row["transcript"])